# Lecke 10 - AI-ügynökök éles környezetben

Ebben a leckében megismered az AI-ügynökök számára a Microsoft Agent Framework (Python) használatával alkalmazható **éles környezeti mintákat**. A következők szerepelnek:

- **Megfigyelhetőség** — időmérés és naplózás hozzáadása az ügynökinterakciókhoz
- **Értékelés** — egy értékelő ügynök használata a válaszok minőségének pontozására
- **Költségkezelés** — stratégiák a tokenoptimalizálásra és a modellválasztásra

A példa egy **utazási ügynök**, amely segít a felhasználóknak utazásokat tervezni, és amelyhez monitorozás és értékelés is kapcsolódik.


## Beállítás


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Üzembe helyezési szempontok

Az AI-ügynökök prototípusokból éles környezetbe való áthelyezése három pillérre való gondos odafigyelést igényel:

1. **Megfigyelhetőség** — Láthatóságra van szükség arra vonatkozóan, hogy mit csinál az ügynök, mennyi ideig tart, és mely eszközöket hívja. Nyomkövetés és naplózás nélkül az éles környezetben jelentkező problémák hibakeresése szinte lehetetlen.

2. **Értékelés** — Az automatizált minőségellenőrzések biztosítják, hogy az ügynök válaszai idővel pontosak, teljesek és hasznosak maradjanak. Egy értékelő ügynök pontozhatja a válaszokat a meghatározott kritériumok alapján.

3. **Költségkezelés** — A tokenok használata közvetlenül befolyásolja a költségeket. Olyan stratégiák, mint a prompt optimalizálása, a modell kiválasztása és a gyorsítótárazás segítenek a kiadások kordában tartásában anélkül, hogy a minőséget feláldoznánk.


## Megfigyelhető ügynök létrehozása

Definiáljuk az utazási eszközöket, és az ügynökhívást időméréssel becsomagoljuk, hogy figyelemmel kísérhessük a késleltetést. Éles környezetben az OpenTelemetry-vel vagy egy hasonló nyomkövető háttérrendszerrel kellene integrálni.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Értékelési minták

Egy gyakori megoldás a gyakorlatban az, hogy egy második ügynököt használnak **értékelőként**. Az értékelő pontozza az elsődleges ügynök válaszát előre meghatározott kritériumok alapján, mint például teljesség, pontosság és segítőkészség.

Ez lehetővé teszi:
- Automatizált minőségi ellenőrzések mielőtt a válaszok elérnék a felhasználókat
- Regresszió észlelése amikor a promptok vagy modellek változnak
- Az ügynök teljesítményének folyamatos nyomon követése az idő során


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Költségkezelési stratégiák

A költségek ellenőrzése kritikus a termelési AI-ügynökök számára. Íme a főbb stratégiák:

| Stratégia | Leírás |
|---|---|
| **Prompt optimalizálás** | Tartsa a rendszerutasításokat tömören. Távolítsa el a felesleges kontextust a bemeneti tokenek csökkentése érdekében. |
| **Modellválasztás** | Használjon kisebb, olcsóbb modelleket (pl. GPT-4o-mini) egyszerű feladatokhoz, például osztályozáshoz vagy kivonatoláshoz, és tartogassa a nagyobb modelleket összetett következtetésekhez. |
| **Gyorsítótárazás** | Tárolja gyorsítótárban az eszközök eredményeit és a gyakori lekérdezéseket, hogy elkerülje a felesleges API-hívásokat. |
| **Token költségvetés** | Állítson be `max_tokens` korlátokat, hogy megakadályozza a váratlanul hosszú válaszokat. |
| **Csoportosítás** | Csoportosítsa több felhasználói lekérdezést egyetlen API-hívásba, ahol lehetséges. |

Gyakorlatban jól működik egy rétegzett megközelítés: irányítsa az egyszerű kéréseket egy gyors, olcsó modellhez, és csak a bonyolultabb lekérdezéseket emelje át egy nagyobb képességű modellhez.


## Összegzés

Ebben a leckében megtanultad, hogyan:

1. **Megfigyelhetőséget hozzáadni** az ügynökök interakcióihoz időzítéssel és naplózással, megteremtve az alapot a nyomon követéshez és a monitorozáshoz.
2. **Értékelni az ügynökök válaszait** automatikusan egy értékelő ügynök használatával, amely pontozza a teljességet, pontosságot és hasznosságot.
3. **Költségek kezelése** prompt optimalizálással, modellválasztással, gyorsítótárazással és token költségvetésekkel.

Ezek a gyártási minták segítenek biztosítani, hogy az AI ügynökeid megbízhatóak, mérhetők és költséghatékonyak legyenek nagy léptékben.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Felelősségkizárás:
Ezt a dokumentumot az AI-alapú fordítószolgáltatás, a Co-op Translator (https://github.com/Azure/co-op-translator) segítségével fordítottuk le. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő irányadónak. Kritikus információk esetén szakmai, emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
